# Fase 3 — execução completa (todos os tópicos e subtópicos)

Notebook curto que só roda o pipeline pros 40 subtópicos, reaproveitando tudo
que o piloto (`pipeline_mcq_fase3.ipynb`) já calibrou e já processou: o
`out_dir` é o mesmo (`saida_fase3/`), então o cache de facetas, a varredura do
corpus, os embeddings, os codebooks e o repositório (com as 249 questões do
piloto já recuperadas) continuam valendo — este notebook **estende**, não
recomeça.

Pré-requisito: já ter rodado o piloto pelo menos até a calibração do limiar de
entropia (§14 de `pipeline_mcq_fase3.ipynb`). Se `saida_fase3/estado/` não
tiver os subtópicos do piloto, a célula de calibração avisa e cai no default
do `Config` (0,010 — já bem próximo do que o piloto calibrou, 0,009-0,012).

**Custo do que este notebook faz de novo**, em relação ao piloto: facetas para
os subtópicos que faltam (chamadas leves, com cache — pula os que já têm
faceta extraída), **uma nova varredura do corpus** (a assinatura inclui o
conjunto de facetas, então passar de 3 pra 40 subtópicos sempre dispara uma
nova passada, ~8 min com os três arquivos), e a execução em si, agora
proporcional a 40 subtópicos em vez de 3 — é a célula de execução completa que
concentra a maior parte do custo em chamadas de LLM. Se quiser ver
comportamento/custo antes de soltar nos 40, rode uma vez com `SUBTOPICOS`
reduzido a um punhado de subtópicos novos.

## 1. Setup

In [18]:
import sys, json, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

AQUI = Path.cwd()                      # pipeline/fase_3
RAIZ = (AQUI / ".." / "..").resolve()  # raiz do repositório
sys.path.insert(0, str(AQUI))
sys.path.insert(0, str(RAIZ))

import utils_fase3 as U
import prompts_fase3 as P
import seed_fase3 as S
from topicos import TOPICOS
from azure_openai_backend import AzureOpenAIBackend

pd.set_option("display.max_colwidth", 90)
print(f"raiz: {RAIZ}")
print(f"tópicos: {len(TOPICOS)} · subtópicos: {sum(len(v) for v in TOPICOS.values())}")

raiz: /home/gbs2/MCQ_CONTEXT_GEN
tópicos: 10 · subtópicos: 40


In [19]:
cfg = U.Config(
    raiz=RAIZ,
    corpus_dir=RAIZ / "dataset" / "corpus-SemProcessamento-publico-PetrolesCompleto",
    out_dir=AQUI / "saida_fase3",   # MESMO out_dir do piloto — reaproveita tudo já calculado
    modelo_forte="gpt-5-4-petrobras",
    modelo_leve="gpt-5-mini-petrobras",
    # limiar_ganho_entropia fica no default do Config (0.010) até a célula de
    # calibração logo abaixo, que lê o histórico do piloto e ajusta se achar
    # dado suficiente.
)
print(cfg.resumo())

{
  "raiz": "/home/gbs2/MCQ_CONTEXT_GEN",
  "corpus_dir": "/home/gbs2/MCQ_CONTEXT_GEN/dataset/corpus-SemProcessamento-publico-PetrolesCompleto",
  "corpus_txt": [],
  "corpus_zip": null,
  "corpus_ignorar": [],
  "out_dir": "/home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3",
  "modelo_forte": "gpt-5-4-petrobras",
  "modelo_leve": "gpt-5-mini-petrobras",
  "modelo_embedding": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "judge_reasoning_effort": "low",
  "gerador_reasoning_effort": "medium",
  "n_facetas_max": 6,
  "alvo_palavras_trecho": 250,
  "sobreposicao_trecho": 0.5,
  "max_linhas_trecho": 120,
  "min_palavras": 80,
  "max_palavras": 450,
  "max_frac_number": 0.15,
  "top_k_lexical": 300,
  "peso_termo_forte": 3.0,
  "peso_termo_apoio": 1.0,
  "cap_termos": 6,
  "n_trechos_por_documento": 6,
  "n_documentos_por_faceta": 4,
  "peso_lexical": 0.4,
  "peso_semantico": 0.6,
  "mmr_lambda": 0.7,
  "mmr_dup_threshold": 0.92,
  "max_palavras_documento": 900,
  "

In [20]:
cfg.judge_reasoning_effort

'low'

In [21]:
# Um deployment por papel. O backend cacheia em disco por hash do pedido, então
# reexecutar o notebook não repaga as chamadas idênticas.
llm_leve = AzureOpenAIBackend(deployment=cfg.modelo_leve, max_tokens=3000)
llm_gerador = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=12000,
                                 reasoning_effort=cfg.gerador_reasoning_effort)
llm_judge = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=8000,
                               reasoning_effort=cfg.judge_reasoning_effort)
# O refinador é o mesmo gpt-5 do gerador — reescrever questão é tarefa de autor.
llm_refinador = llm_gerador

llm_leve.doctor()

Configuração (segredos mascarados):
  azure_endpoint: https://api***.petrobras.com.br/ia/openai/v1/openai-azure/openai
  azure_api_key: <definida:72ce953d>
  azure_api_version: 2024-10-21
  default_deployment: None
  ca_bundle: /home/gbs2/MCQ_CONTEXT_GEN/petrobras-ca-root.pem
  use_base_url: True
  dotenv_path: /home/gbs2/MCQ_CONTEXT_GEN/.env
  dotenv_found: True
  config_ini: None
  placeholders: []
  deployment: gpt-5-mini-petrobras (reasoning=True)
Fazendo uma chamada de teste...
  resposta: 'ok'
OK.


## 2. Insumos: banco seed e escopo completo

In [22]:
DIR_FASE2 = RAIZ / "pipeline" / "fase_2" / "questionarios" / "respostas_questionario"

banco_seed = S.construir_banco_seed(DIR_FASE2, min_nota_humana=0.5)
S.salvar_banco_seed(banco_seed, cfg.out_dir / "banco_seed.jsonl")
print(f"{len(banco_seed)} questões no banco seed")

banco seed: 22 questões de 30 pares (2 avaliadores)
  por condição: {'instruction-only': 12, 'context-grounded': 10}
  por nota humana: {1.0: 6, 0.5: 16}
  descartados (empate ou sem voto decisivo): 8
22 questões no banco seed


In [23]:
# Todos os tópicos/subtópicos — a única diferença de escopo em relação ao
# piloto (que rodava só 3) é esta linha.
SUBTOPICOS = [(t, s) for t, subs in TOPICOS.items() for s in subs]
print(f"{len(SUBTOPICOS)} subtópicos em {len(TOPICOS)} tópicos")

40 subtópicos em 10 tópicos


## 3. Limiar de entropia calibrado no piloto

Lê o histórico de entropia dos subtópicos que já rodaram (salvo em
`saida_fase3/estado/`) e sugere o limiar do critério de parada, do jeito que o
notebook principal faz na §14 (`U.sugerir_limiar_entropia`). Se não achar
nenhum estado ainda, mantém o default do `Config` (0,010) e avisa.

Ressalva do bug de perda de dados de ago/2026 (ver memória do projeto): o
histórico do piloto tem um ponto de transição — a rodada em que o repositório
foi recuperado — que é um outlier no ganho de entropia. A função usa percentil
75, então é razoavelmente robusta a esse único ponto, mas o número calibrado
(0,009-0,012 na última checagem) é uma referência, não uma verdade absoluta.

In [24]:
historicos = {}
for p in glob.glob(str(cfg.out_dir / "estado" / "*.json")):
    est = json.loads(Path(p).read_text(encoding="utf-8"))
    if len(est.get("historico_entropia", [])) >= 3:
        historicos[est["subtopico"]] = est["historico_entropia"]

if historicos:
    limiar = U.sugerir_limiar_entropia(historicos, percentil=75)
    cfg.limiar_ganho_entropia = limiar
    print(f"\nadotando limiar calibrado: {cfg.limiar_ganho_entropia:.4f}")
else:
    print(f"nenhum estado de piloto encontrado em {cfg.out_dir / 'estado'} — "
          f"mantendo o default do Config: {cfg.limiar_ganho_entropia:.4f}")

ganhos de entropia por fase:
  crescimento  n=10   mediana +0.0170 · p75 +0.0904 · máx +0.1864
  platô        n=47   mediana +0.0000 · p75 +0.0116 · máx +0.2908
limiar sugerido (p75 do platô, travado em 0): 0.0116

adotando limiar calibrado: 0.0116


## 4. Facetas, varredura do corpus, embeddings e planos — para TODOS os subtópicos

A varredura do corpus (célula seguinte) é a única etapa cara que roda de novo
inteira: a assinatura inclui o conjunto de facetas, e o conjunto mudou de 3
pra até 40 subtópicos.

In [25]:
facetas = U.extrair_facetas(llm_leve, TOPICOS, cfg, subtopicos_alvo=SUBTOPICOS)
por_sub = U.agrupar_por_subtopico(facetas)
print(f"\n{len(facetas)} facetas em {len(por_sub)} subtópicos")


238 facetas em 39 subtópicos


In [26]:
%%time
U.descrever_corpus(cfg)
U.varrer_corpus(facetas, cfg)          # ~8 min com os três arquivos do corpus
candidatos = U.carregar_candidatos(cfg)
print(f"\n{sum(len(v) for v in candidatos.values()):,} candidatos em {len(candidatos)} facetas")

corpus: 2 arquivo(s), 0.97 GB, ~6M linhas
  corpusPublico(sem IBICT)-SemProcessamento.txt      0.54 GB · ~   3.4M linhas ·  25.2 palavras/linha ·   57% das linhas
  corpusPublicoIBICT-SemProcessamento.txt            0.43 GB · ~   2.6M linhas ·  26.1 palavras/linha ·   43% das linhas
índice já existe e confere (/home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3/indices/candidatos.jsonl) — reaproveitando

32,436 candidatos em 223 facetas
CPU times: user 2.42 s, sys: 767 ms, total: 3.19 s
Wall time: 4.25 s


In [27]:
emb = U.Embedder(cfg)

planos = {}
for subtopico, fs in por_sub.items():
    planos[subtopico] = U.plano_de_documentos(fs, candidatos, emb, cfg)
print(f"planos montados para {len(planos)} subtópicos · "
      f"{sum(len(p) for p in planos.values())} documentos no total")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 520.10it/s]
/home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/utils_fase3.py:689: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


embedder: paraphrase-multilingual-MiniLM-L12-v2 (dim=384) · cache com 4603 vetores
    planejamento e controle operacional                      4 cand -> 1 doc(s)
    gestao de indicadores e metas                           23 cand -> 4 doc(s)
    eficiencia e disponibilidade de sistemas               176 cand -> 4 doc(s)
    confiabilidade e integridade de ativos                  41 cand -> 4 doc(s)
    gestao de riscos e seguranca operacional               300 cand -> 4 doc(s)
    custos, perdas de producao e melhoria continua         300 cand -> 4 doc(s)
    gestao do ciclo de vida e extensao de vida util         20 cand -> 3 doc(s)
    degradacao, integridade e fitness for service          300 cand -> 4 doc(s)
    estrategia e logisticao de descomissionamento (estal     0 cand -> 0 doc(s)
    preparacao, parada definitiva e limpeza de sistemas     24 cand -> 4 doc(s)
    interfaces subsea e descomissionamento de poços         19 cand -> 3 doc(s)
    gestao de residuos, ambiental e s

## 5. Tolerâncias de vício e codebooks de entropia — para TODOS os subtópicos

In [28]:
sugerido = U.calibrar_tolerancias(banco_seed, emb, cfg, percentil=75)
cfg.tol_similaridade = sugerido["similaridade"]
cfg.tol_comprimento  = sugerido["comprimento"]
cfg.tol_distratores  = sugerido["distratores"]

scorer de vícios sobre 22 questões de referência

  vício           mediana    p75    p90    | tolerância atual  reprova
  similaridade       0.34   0.54   0.67                  0.55     27%
  comprimento        0.47   0.65   1.00                  0.55     41%
  distratores        0.00   0.33   0.63                  0.50     14%
  racionalizacao     0.00   0.00   0.00                  0.00      5%

  custo de cada tolerância candidata (fração que iria para o refinador):
    similaridade   p50/p75/p90: 0.34 -> 50% · 0.54 -> 27% · 0.67 -> 9%
    comprimento    p50/p75/p90: 0.47 -> 50% · 0.65 -> 27% · 1.0 -> 0%
    distratores    p50/p75/p90: 0.0 -> 45% · 0.33 -> 45% · 0.63 -> 14%
    racionalizacao p50/p75/p90: 0.0 -> 5% · 0.0 -> 5% · 0.0 -> 5%

  sugestão (p75): {'similaridade': 0.544, 'comprimento': 0.654, 'distratores': 0.333, 'racionalizacao': 0.0}
  Aperte em relação a isso se quiser que a fase 3 melhore o viés
  da fase 2 em vez de reproduzi-lo — pagando mais refinamento.


In [29]:
codebooks = {}
for subtopico, fs in por_sub.items():
    codebooks[subtopico] = U.codebook_do_subtopico(subtopico, fs, candidatos, emb, cfg)
print(f"{len(codebooks)} codebooks prontos")

  codebook 'sub::Geral' já treinado (k=24) — reaproveitando
  codebook 'sub::Medição Fiscal de óleo, medição fis' já treinado (k=24) — reaproveitando
  codebook 'sub::Produtos químicos' já treinado (k=6) — reaproveitando
  codebook 'sub::Tratamento de água produzida para d' já treinado (k=24) — reaproveitando
  codebook 'sub::Sistema de captação e tratamento de' já treinado (k=24) — reaproveitando
  codebook 'sub::Sistema de drenagem aberta e fechad' já treinado (k=24) — reaproveitando
  codebook 'sub::Operação Geral FPSO' já treinado (k=24) — reaproveitando
  codebook 'sub::Produção, processo e processamento ' já treinado (k=24) — reaproveitando
  codebook 'sub::Sistema de Alívio e Flare' já treinado (k=24) — reaproveitando
  codebook 'sub::Operação de Tratadores Eletrostátic' já treinado (k=24) — reaproveitando
  codebook 'sub::Operação de Separadores Trifásicos' já treinado (k=24) — reaproveitando
  codebook 'sub::Tratamento do Gás produzido' já treinado (k=24) — reaproveitando
  co

## 6. Repositório, pool de few-shot e execução completa

In [30]:
repo = U.Repositorio(cfg, emb)
pool = U.PoolFewShot(banco_seed, repo, cfg)
print(f"repositório: {len(repo)} questões já armazenadas (piloto incluso)")
print("pool de few-shot:", pool.composicao())

repositório: 249 questões já armazenadas (piloto incluso)
pool de few-shot: {'seed': 22, 'aprovadas_alto_score': 249, 'peso_do_seed': 0.1}


`executar_subtopico` encadeia geração → vícios → refinamento → judge →
armazenamento → entropia (judge por último, ver memória do projeto), avança de
documento quando o subtópico satura e termina quando a fila de documentos
acaba. É retomável: os subtópicos do piloto já processados continuam de onde
pararam (a checagem de consistência do início de `executar_subtopico` confere
isso antes de seguir); os subtópicos novos começam do zero.

`max_rodadas` fica no default da própria função (60) — quem decide quando cada
subtópico termina é o critério de parada por entropia, não esse teto (ele só
evita um loop sem fim). `repo.salvar()` a cada subtópico concluído, não só no
final — com 40 subtópicos essa célula pode rodar por muito tempo, e as
questões em si já são gravadas incrementalmente (`Repositorio.adicionar`)
conforme aprovadas, mas os embeddings pesam menos gravados com mais frequência.

> Célula cara. Considere reduzir `SUBTOPICOS` a um punhado antes de soltar nos
> 40, se ainda não tiver visto o comportamento/custo desta versão do pipeline
> (judge no fim do loop) em produção.

In [31]:
%%time
estados = {}
for topico, subtopico in SUBTOPICOS:
    estados[subtopico] = U.executar_subtopico(
        subtopico=subtopico, topico=topico,
        facetas=por_sub[subtopico], plano=planos[subtopico],
        llm_leve=llm_leve, llm_forte=llm_gerador, llm_judge=llm_judge,
        pool=pool, repo=repo, emb=emb, codebook=codebooks[subtopico],
        cfg=cfg,
    )
    repo.salvar()

print(f"\nrepositório: {len(repo)} questões · pool: {pool.composicao()}")


=== Geral === (35 documentos, retomando na rodada 1, doc 0)
    rodada  1 · planejamento e controle operaciona gerou  6 · refinou 1 · descartou 2 · judge  4 · guardou 4
       entropia 0.3272 (ganho +0.3272, aquecendo) · pool 4 · estagnadas 0
    rodada  2 · planejamento e controle operaciona gerou  6 · refinou 1 · descartou 1 · judge  5 · guardou 5
       entropia 0.4124 (ganho +0.0853, aquecendo) · pool 9 · estagnadas 0
    rodada  3 · planejamento e controle operaciona gerou  6 · refinou 3 · descartou 0 · judge  6 · guardou 6
       entropia 0.3649 (ganho -0.0476, sem ganho) · pool 15 · estagnadas 1
    rodada  4 · planejamento e controle operaciona gerou  6 · refinou 0 · descartou 3 · judge  3 · guardou 3
       entropia 0.3524 (ganho -0.0125, sem ganho) · pool 18 · estagnadas 2
    rodada  5 · planejamento e controle operaciona gerou  6 · refinou 3 · descartou 0 · judge  5 · guardou 5
       entropia 0.3737 (ganho +0.0213, ganho) · pool 23 · estagnadas 0
    rodada  6 · planejame

LLMError: chamada ao Azure falhou (deployment='gpt-5-4-petrobras', reasoning=True): Request timed out.

## 7. Export final

In [32]:
destino = cfg.out_dir / "questoes_fase3.jsonl"
with open(destino, "w", encoding="utf-8") as fh:
    for q in repo.questoes:
        fh.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(repo)} questões -> {destino}")
print(f"embeddings           -> {repo.caminho_emb}")
print(f"estado por subtópico -> {cfg.out_dir / 'estado'}")
print(f"log de rodadas       -> {cfg.out_dir / 'logs' / 'rodadas.jsonl'}")
print(f"\nuso de tokens:")
for nome, llm in [("leve", llm_leve), ("gerador", llm_gerador), ("judge", llm_judge)]:
    u = llm.usage
    print(f"  {nome:<8} {u.calls:>4} chamadas ({u.cached_calls} de cache) · "
          f"{u.prompt_tokens:>9,} in · {u.completion_tokens:>8,} out")

493 questões -> /home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3/questoes_fase3.jsonl
embeddings           -> /home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3/repositorio/embeddings.npy
estado por subtópico -> /home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3/estado
log de rodadas       -> /home/gbs2/MCQ_CONTEXT_GEN/pipeline/fase_3/saida_fase3/logs/rodadas.jsonl

uso de tokens:
  leve        9 chamadas (1 de cache) ·    21,061 in ·   12,172 out
  gerador   272 chamadas (0 de cache) ·   645,112 in ·  437,981 out
  judge      54 chamadas (0 de cache) ·   103,463 in ·   38,881 out


## 8. O que revisar antes de considerar o lote pronto

1. os documentos consolidados que começam com `AVISO:` — recuperação fraca
   naquela faceta, e questão gerada dali não presta;
2. a taxa de descarte do judge por subtópico — descarte alto costuma ser
   documento ruim, não gerador ruim;
3. a distribuição de dificuldade — se o judge estiver marcando quase tudo como
   "media", a escala não está discriminando;
4. rodar de novo `analysis/avaliacao_dificuldade_ollama.ipynb` sobre o lote
   completo, pra comparar a fração de questões "Fácil" com a 1ª rodada (87%).